In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)

In [2]:
df = sns.load_dataset("titanic")
data = df[["survived", "pclass", "sex", "age", "fare", "sibsp", "parch"]].copy()
data["is_female"] = (data["sex"] == "female").astype(int)
X = data[["pclass", "is_female", "age", "fare", "sibsp", "parch"]]
y = data["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LogisticRegression(max_iter=1000))
])
pipe.fit(X_train, y_train)
p = pipe.predict_proba(X_test)[:, 1] # probability of survived
pred = (p >= 0.5).astype(int) # class at threshold 0.5

In [3]:
# 1. confusion matrix
cm = confusion_matrix(y_test, pred)
print(cm)

tn, fp, fn, tp = cm.ravel()
print(tn, fp, fn, tp)

[[95 15]
 [20 49]]
95 15 20 49


In [4]:
# 2. metrics BY HAND from the four numbers
print("Accuracy:   ", round((tp + tn)/ (tn + fp + fn + tp), 4))
print("Precision:  ", round((tp/ (tp + fp)), 4))
print("Recall:     ", round((tp / (tp + fn)), 4))
print("F1:         ", round(2 * tp / (2 * tp + fp + fn), 4))

Accuracy:    0.8045
Precision:   0.7656
Recall:      0.7101
F1:          0.7368


In [5]:
# 3. the same with sklearn (must match)
print(round(accuracy_score(y_test, pred), 4), round(precision_score(y_test, pred), 4),
      round(recall_score(y_test, pred), 4), round(f1_score(y_test, pred), 4))

0.8045 0.7656 0.7101 0.7368


In [7]:
# 4. the threshold trade-off
for t in (0.3, 0.5, 0.7):
    pr = (p >= t).astype(int)
    print(t, confusion_matrix(y_test, pr).ravel(), 
          round(precision_score(y_test, pr), 4),
          round(recall_score(y_test, pr), 4),
          round(f1_score(y_test, pr), 4))

0.3 [83 27 14 55] 0.6707 0.7971 0.7285
0.5 [95 15 20 49] 0.7656 0.7101 0.7368
0.7 [108   2  38  31] 0.9394 0.4493 0.6078


In [8]:
# 5. ROC-AUC uses the PROBABILITIES, not the 0/1 predictions
print("Auc: ", round(roc_auc_score(y_test, p), 4))

Auc:  0.8515


In [9]:
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier

In [13]:
Xi, yi = make_classification(n_samples=10000, n_features=10, n_informative=5,
                             weights=[0.99, 0.01], flip_y=0, class_sep=1.0, random_state=42)
Xa, Xb, ya, yb = train_test_split(
    Xi, yi, test_size=0.2, random_state=42, stratify=yi
)
print("Fraud cases in test: ", yb.sum(), "of", len(yb))

Fraud cases in test:  20 of 2000


In [16]:
dummy = DummyClassifier(strategy="most_frequent").fit(Xa, ya)
pd_ = dummy.predict(Xb)
print("Predict:", accuracy_score(yb, pd_), "recall", recall_score(yb, pd_))

Predict: 0.99 recall 0.0


In [17]:
m = LogisticRegression(max_iter=1000).fit(Xa, ya)
pm = m.predict(Xb)
print("logreg acc", accuracy_score(yb, pm), "recall", recall_score(yb, pm))
print(confusion_matrix(yb, pm).ravel())

logreg acc 0.991 recall 0.1
[1980    0   18    2]


In [18]:
for t in (0.5, 0.05):
    pr = ((m.predict_proba(Xb)[:, 1] >= t).astype(int))
    print(t, confusion_matrix(yb, pr).ravel(),
          round(precision_score(yb, pr, zero_division=0), 3), 
          round(recall_score(yb, pr), 3))

0.5 [1980    0   18    2] 1.0 0.1
0.05 [1924   56   14    6] 0.097 0.3
